# Notebook 07: Neural Network Training & Evaluation

**Purpose:** Train and evaluate deep learning models for angle grinder detection

**Objectives:**
1. Load neural features from notebook 04 (1ch or 3ch spectrograms)
2. Train multiple neural architectures (MLP, CNN-1D, LSTM, CNN-LSTM)
3. Compare performance vs model size/speed trade-offs
4. Compare with best classical models from notebook 06a
5. Evaluate deployment feasibility for ESP32-S3

**Key Constraints:**
- Target device: ESP32-S3 (240MHz dual-core, ~500KB RAM)
- Model size: <500KB after quantization
- Inference time: <50ms
- Maintain high recall (minimize false negatives)

**Memory Management:**
- Active cleanup after each model training
- Keras session clearing to free GPU/RAM

---

## Section 1: Setup & Configuration

In [1]:
import os, sys
from pathlib import Path
import json
import time
from datetime import datetime
import warnings
import gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras import backend as K

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')
print(f'✓ GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('✓ All libraries imported')

TensorFlow version: 2.17.1
Keras version: 3.10.0
✓ GPU Available: False
✓ All libraries imported


In [5]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features' / 'neural'
MODELS_DIR = PROJECT_ROOT / 'models' / 'neural'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Features dir: {FEATURES_DIR}')
print(f'Models dir:   {MODELS_DIR}')
print(f'Results dir:  {RESULTS_DIR}')

Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv5
Features dir: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/neural
Models dir:   /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/neural
Results dir:  /Users/harryirving/Development/projects/ai-ml/BikeAIv5/results


In [6]:
# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# TensorFlow configuration
# Limit GPU memory growth if GPU available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'✓ GPU memory growth enabled for {len(gpus)} GPU(s)')
    except RuntimeError as e:
        print(f'GPU configuration error: {e}')

print(f'✓ Random seed set to {SEED}')

✓ Random seed set to 42


## Section 2: Load Neural Features & Prepare Data

In [7]:
print('\nLOADING NEURAL FEATURES FROM NOTEBOOK 04...')
print('='*70)

# Load spectrogram features from notebook 04
spec_1ch_file = FEATURES_DIR / 'spectrograms_1ch.npy'
spec_3ch_file = FEATURES_DIR / 'spectrograms_3ch.npy'
labels_file = FEATURES_DIR / 'labels.npy'
metadata_file = FEATURES_DIR / 'spectrogram_metadata.json'

# Check which files exist
print(f'\nChecking for spectrogram files...')
print(f'  1-channel: {"✓" if spec_1ch_file.exists() else "✗"} {spec_1ch_file.name}')
print(f'  3-channel: {"✓" if spec_3ch_file.exists() else "✗"} {spec_3ch_file.name}')
print(f'  Labels:    {"✓" if labels_file.exists() else "✗"} {labels_file.name}')

# Load metadata
if metadata_file.exists():
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    print(f'\n✓ Loaded meta')
    print(f'  Total samples: {metadata["total_samples"]:,}')
    print(f'  Parameters: sr={metadata["parameters"]["sr"]}, n_mels={metadata["parameters"]["n_mels"]}')

# Decide which features to use
# 3-channel is preferred (mel + delta + delta-delta) for better performance
# 1-channel is smaller and faster for deployment

USE_3CH = True  # Set to False to use 1-channel (faster, smaller models)

if USE_3CH and spec_3ch_file.exists():
    print(f'\n📊 Using 3-channel spectrograms (mel + delta + delta-delta)...')
    X_spec = np.load(spec_3ch_file)
    y = np.load(labels_file)
    feature_type = '3ch'
elif spec_1ch_file.exists():
    print(f'\n📊 Using 1-channel spectrograms (mel only)...')
    X_spec = np.load(spec_1ch_file)
    y = np.load(labels_file)
    feature_type = '1ch'
else:
    raise FileNotFoundError('Spectrogram features not found. Run notebook 04 first.')

print(f'\n✓ Loaded {feature_type} spectrogram features:')
print(f'  Feature shape: {X_spec.shape}')
print(f'  Label shape:   {y.shape}')
print(f'  Memory usage:  {X_spec.nbytes / 1024**2:.1f} MB')
print(f'\n  Grinder samples:     {(y == 1).sum():,}')
print(f'  Non-grinder samples: {(y == 0).sum():,}')
print(f'  Class balance:       {(y == 1).sum() / len(y) * 100:.1f}% grinder')

if len(X_spec.shape) == 4:
    print(f'\n  Shape interpretation: ({X_spec.shape[0]} samples, {X_spec.shape[1]} time, {X_spec.shape[2]} freq, {X_spec.shape[3]} channels)')
elif len(X_spec.shape) == 3:
    print(f'\n  Shape interpretation: ({X_spec.shape[0]} samples, {X_spec.shape[1]} time, {X_spec.shape[2]} freq)')

print('='*70)


LOADING NEURAL FEATURES FROM NOTEBOOK 04...

Checking for spectrogram files...
  1-channel: ✓ spectrograms_1ch.npy
  3-channel: ✓ spectrograms_3ch.npy
  Labels:    ✓ labels.npy

✓ Loaded meta
  Total samples: 60,348
  Parameters: sr=16000, n_mels=40

📊 Using 3-channel spectrograms (mel + delta + delta-delta)...

✓ Loaded 3ch spectrogram features:
  Feature shape: (60348, 101, 40, 3)
  Label shape:   (60348,)
  Memory usage:  2790.1 MB

  Grinder samples:     32,938
  Non-grinder samples: 27,410
  Class balance:       54.6% grinder

  Shape interpretation: (60348 samples, 101 time, 40 freq, 3 channels)


In [8]:
print('\nPREPARING DATA SPLITS...')
print('='*70)

# Split: 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_spec, y, test_size=0.15, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
)

print(f'Train set: {X_train.shape[0]:,} samples')
print(f'Val set:   {X_val.shape[0]:,} samples')
print(f'Test set:  {X_test.shape[0]:,} samples')

print(f'\nTrain class balance: {(y_train == 1).sum() / len(y_train) * 100:.1f}% grinder')
print(f'Val class balance:   {(y_val == 1).sum() / len(y_val) * 100:.1f}% grinder')
print(f'Test class balance:  {(y_test == 1).sum() / len(y_test) * 100:.1f}% grinder')

# Calculate class weights for imbalanced data
n_grinder = (y_train == 1).sum()
n_non_grinder = (y_train == 0).sum()
total = len(y_train)

class_weight = {
    0: total / (2 * n_non_grinder),
    1: total / (2 * n_grinder)
}

print(f'\nClass weights (for training):')
print(f'  Non-grinder (0): {class_weight[0]:.3f}')
print(f'  Grinder (1):     {class_weight[1]:.3f}')

# Free original X_spec to save memory
del X_spec, X_temp
gc.collect()
print(f'\n✓ Freed original arrays from memory')

print('='*70)


PREPARING DATA SPLITS...
Train set: 42,267 samples
Val set:   9,028 samples
Test set:  9,053 samples

Train class balance: 54.6% grinder
Val class balance:   54.6% grinder
Test class balance:  54.6% grinder

Class weights (for training):
  Non-grinder (0): 1.101
  Grinder (1):     0.916

✓ Freed original arrays from memory


In [9]:
print('\nNORMALIZATION...')
print('='*70)

# Inspect shape
original_shape = X_train.shape
print(f'Original shape: {original_shape}')

# For spectrograms (3D or 4D), normalize per sample
if len(X_train.shape) in (3, 4):
    print('Applying per-sample normalization for spectrograms...')

    X_train_norm = np.zeros_like(X_train, dtype=np.float32)
    X_val_norm   = np.zeros_like(X_val,   dtype=np.float32)
    X_test_norm  = np.zeros_like(X_test,  dtype=np.float32)

    def norm_sample(x):
        mean = x.mean()
        std = x.std() + 1e-8
        return (x - mean) / std

    for i in range(len(X_train)):
        X_train_norm[i] = norm_sample(X_train[i])

    for i in range(len(X_val)):
        X_val_norm[i] = norm_sample(X_val[i])

    for i in range(len(X_test)):
        X_test_norm[i] = norm_sample(X_test[i])

    print('✓ Per-sample normalization complete')

elif len(X_train.shape) == 2:
    print('Applying StandardScaler for 1D features...')

    scaler = StandardScaler()
    X_train_norm = scaler.fit_transform(X_train)
    X_val_norm   = scaler.transform(X_val)
    X_test_norm  = scaler.transform(X_test)

    print('✓ StandardScaler normalization complete')

    import joblib
    scaler_path = MODELS_DIR / 'scaler_neural.pkl'
    joblib.dump(scaler, scaler_path)
    print(f'✓ Saved scaler: {scaler_path.name}')

else:
    raise ValueError(f'Unexpected input shape for normalization: {X_train.shape}')

print(f'\nNormalized shapes:')
print(f'  Train: {X_train_norm.shape}')
print(f'  Val:   {X_val_norm.shape}')
print(f'  Test:  {X_test_norm.shape}')

# Free non-normalized arrays
del X_train, X_val, X_test
gc.collect()
print(f'\n✓ Freed non-normalized arrays from memory')

print('='*70)


NORMALIZATION...
Original shape: (42267, 101, 40, 3)
Applying per-sample normalization for spectrograms...
✓ Per-sample normalization complete

Normalized shapes:
  Train: (42267, 101, 40, 3)
  Val:   (9028, 101, 40, 3)
  Test:  (9053, 101, 40, 3)

✓ Freed non-normalized arrays from memory


## Section 3: Define Neural Network Architectures

### 3.1 Multi-Layer Perceptron (MLP) - Baseline

In [10]:
def build_mlp(input_shape, dropout_rate=0.3, l2_reg=0.001):
    """
    Build a Multi-Layer Perceptron classifier.
    
    Simple feedforward network - good baseline for comparison.
    Works on flattened input.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Flatten if needed
        layers.Flatten(),
        
        # Dense layers
        layers.Dense(256, activation='relu', 
                    kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(128, activation='relu',
                    kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.Dense(64, activation='relu',
                    kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        
        # Output
        layers.Dense(1, activation='sigmoid')
    ], name='MLP')
    
    return model

# Test build
print('Testing MLP architecture...')
test_mlp = build_mlp(X_train_norm.shape[1:])
test_mlp.summary()
print(f'\n✓ MLP model parameters: {test_mlp.count_params():,}')
del test_mlp
K.clear_session()

Testing MLP architecture...


Model: "MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 12120)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     3,102,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,145,729 (12.00 MB)

 Trainable params: 3,144,961 (12.00 MB)

 Non-trainable params: 768 (3.00 KB)


✓ MLP model parameters: 3,145,729


### 3.2 1D Convolutional Neural Network (CNN-1D)

In [11]:
def build_cnn1d(input_shape, dropout_rate=0.3, l2_reg=0.001):
    """
    Build a 1D CNN for temporal feature extraction.
    
    Good for learning temporal patterns in spectrograms.
    Input: (time_steps, features)
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # First conv block
        layers.Conv1D(64, kernel_size=5, padding='same', activation='relu',
                     kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate),
        
        # Second conv block
        layers.Conv1D(128, kernel_size=3, padding='same', activation='relu',
                     kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate),
        
        # Third conv block
        layers.Conv1D(64, kernel_size=3, padding='same', activation='relu',
                     kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        
        # Dense layers
        layers.Dense(64, activation='relu',
                    kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        
        # Output
        layers.Dense(1, activation='sigmoid')
    ], name='CNN1D')
    
    return model

print('✓ CNN-1D architecture defined')

✓ CNN-1D architecture defined


### 3.3 LSTM Network

In [12]:
def build_lstm(input_shape, dropout_rate=0.3, l2_reg=0.001):
    """
    Build an LSTM network for sequential pattern learning.
    
    Captures long-term dependencies in audio sequences.
    Input: (time_steps, features)
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # LSTM layers
        layers.LSTM(128, return_sequences=True,
                   kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        layers.LSTM(64,
                   kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        # Dense layers
        layers.Dense(32, activation='relu',
                    kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        
        # Output
        layers.Dense(1, activation='sigmoid')
    ], name='LSTM')
    
    return model

print('✓ LSTM architecture defined')

✓ LSTM architecture defined


### 3.4 Hybrid CNN-LSTM

In [13]:
def build_cnn_lstm(input_shape, dropout_rate=0.3, l2_reg=0.001):
    """
    Build a hybrid CNN-LSTM architecture.
    
    CNN extracts local features, LSTM captures temporal dependencies.
    Often the best of both worlds for audio classification.
    Input: (time_steps, features)
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # CNN feature extraction
        layers.Conv1D(64, kernel_size=5, padding='same', activation='relu',
                     kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate * 0.5),
        
        layers.Conv1D(128, kernel_size=3, padding='same', activation='relu',
                     kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate * 0.5),
        
        # LSTM temporal modeling
        layers.LSTM(64, return_sequences=False,
                   kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        
        # Dense classifier
        layers.Dense(32, activation='relu',
                    kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        
        # Output
        layers.Dense(1, activation='sigmoid')
    ], name='CNN_LSTM')
    
    return model

print('✓ CNN-LSTM hybrid architecture defined')

✓ CNN-LSTM hybrid architecture defined


## Section 4: Training Configuration

In [14]:
# Training hyperparameters
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
PATIENCE = 10  # Early stopping patience

print('Training Configuration:')
print(f'  Batch size:     {BATCH_SIZE}')
print(f'  Max epochs:     {EPOCHS}')
print(f'  Learning rate:  {LEARNING_RATE}')
print(f'  Early stopping: {PATIENCE} epochs patience')
print(f'  Class weights:  Enabled')

Training Configuration:
  Batch size:     32
  Max epochs:     50
  Learning rate:  0.001
  Early stopping: 10 epochs patience
  Class weights:  Enabled


In [15]:
def get_callbacks(model_name):
    """
    Create training callbacks for model.
    """
    # Model checkpoint
    checkpoint_path = MODELS_DIR / f'{model_name}_best.keras'
    checkpoint = callbacks.ModelCheckpoint(
        str(checkpoint_path),
        monitor='val_f1',
        mode='max',
        save_best_only=True,
        verbose=1
    )
    
    # Early stopping
    early_stop = callbacks.EarlyStopping(
        monitor='val_f1',
        mode='max',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    )
    
    # Learning rate reduction
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
    
    return [checkpoint, early_stop, reduce_lr]

print('✓ Callback configuration ready')

✓ Callback configuration ready


In [16]:
# Custom F1 metric for Keras
class F1Score(keras.metrics.Metric):
    def __init__(self, name='f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision = keras.metrics.Precision()
        self.recall = keras.metrics.Recall()
    
    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)
    
    def result(self):
        p = self.precision.result()
        r = self.recall.result()
        return 2 * ((p * r) / (p + r + K.epsilon()))
    
    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()

print('✓ F1 metric defined')

✓ F1 metric defined


## Section 5: Prepare Data for Different Architectures

In [17]:
print('\nPREPARING DATA FOR DIFFERENT ARCHITECTURES...')
print('='*70)

# Prepare inputs for CNN/LSTM (need 2D: time x features)
if len(X_train_norm.shape) == 3:
    # (samples, time, freq) - already good for CNN-1D/LSTM
    X_train_cnn = X_train_norm
    X_val_cnn = X_val_norm
    X_test_cnn = X_test_norm
    cnn_input_shape = (X_train_cnn.shape[1], X_train_cnn.shape[2])
    print(f'3D input detected - using directly for CNN/LSTM')
    print(f'CNN/LSTM input shape: {cnn_input_shape}')

elif len(X_train_norm.shape) == 4:
    # (samples, time, freq, channels) - reshape to (samples, time, freq*channels)
    N_train, T, F, C = X_train_norm.shape
    N_val, _, _, _ = X_val_norm.shape
    N_test, _, _, _ = X_test_norm.shape
    
    X_train_cnn = X_train_norm.reshape(N_train, T, F * C)
    X_val_cnn = X_val_norm.reshape(N_val, T, F * C)
    X_test_cnn = X_test_norm.reshape(N_test, T, F * C)
    cnn_input_shape = (T, F * C)
    print(f'4D input detected - reshaped to (time, freq*channels) for CNN/LSTM')
    print(f'CNN/LSTM input shape: {cnn_input_shape}')

else:
    print('Unexpected shape - only MLP will run')
    X_train_cnn = None

# MLP uses the original normalized data (will flatten internally)
X_train_mlp = X_train_norm
X_val_mlp = X_val_norm
X_test_mlp = X_test_norm
mlp_input_shape = X_train_norm.shape[1:]
print(f'\nMLP input shape: {mlp_input_shape}')

print('='*70)


PREPARING DATA FOR DIFFERENT ARCHITECTURES...
4D input detected - reshaped to (time, freq*channels) for CNN/LSTM
CNN/LSTM input shape: (101, 120)

MLP input shape: (101, 40, 3)


## Section 6: Train Models

In [18]:
# Storage for results
trained_models = {}
training_histories = {}
training_times = {}

### 6.1 Train MLP

In [19]:
print('\nTRAINING MLP MODEL...')
print('='*70)

# Build model
mlp_model = build_mlp(mlp_input_shape)

# Compile
mlp_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        F1Score(name='f1')
    ]
)

print('Model compiled. Starting training...')
start_time = time.time()

# Train
history = mlp_model.fit(
    X_train_mlp, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val_mlp, y_val),
    class_weight=class_weight,
    callbacks=get_callbacks('mlp'),
    verbose=1
)

train_time = time.time() - start_time

# Store
trained_models['MLP'] = mlp_model
training_histories['MLP'] = history.history
training_times['MLP'] = train_time

print(f'\n✓ MLP training complete in {train_time/60:.1f} minutes')
print(f'  Best val F1: {max(history.history["val_f1"]):.4f}')
print('='*70)

# Memory cleanup
del history
gc.collect()


TRAINING MLP MODEL...
Model compiled. Starting training...
Epoch 1/50
1319/1321 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8421 - f1: 0.8523 - loss: 0.8077 - precision: 0.8707 - recall: 0.8348
Epoch 1: val_f1 improved from -inf to 0.84522, saving model to /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/neural/mlp_best.keras
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.8421 - f1: 0.8524 - loss: 0.8073 - precision: 0.8707 - recall: 0.8348 - val_accuracy: 0.8201 - val_f1: 0.8452 - val_loss: 0.7772 - val_precision: 0.7969 - val_recall: 0.8998 - learning_rate: 0.0010
Epoch 2/50
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8869 - f1: 0.8948 - loss: 0.4637 - precision: 0.9085 - recall: 0.8816
Epoch 2: val_f1 did not improve from 0.84522
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8869 - f1: 0.8948 - loss: 0.4637 - precision: 0.9085 - recall: 0.8816 - val_accuracy: 0.5800 - val_f1: 0.3800 - val_loss: 2.4886 - val_precision: 0.9781 - val

1101

### 6.2 Train CNN-1D

In [20]:
if X_train_cnn is not None:
    print('\nTRAINING CNN-1D MODEL...')
    print('='*70)
    
    # Build model
    cnn_model = build_cnn1d(cnn_input_shape)
    
    # Compile
    cnn_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            F1Score(name='f1')
        ]
    )
    
    print('Model compiled. Starting training...')
    start_time = time.time()
    
    # Train
    history = cnn_model.fit(
        X_train_cnn, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_val_cnn, y_val),
        class_weight=class_weight,
        callbacks=get_callbacks('cnn1d'),
        verbose=1
    )
    
    train_time = time.time() - start_time
    
    # Store
    trained_models['CNN1D'] = cnn_model
    training_histories['CNN1D'] = history.history
    training_times['CNN1D'] = train_time
    
    print(f'\n✓ CNN-1D training complete in {train_time/60:.1f} minutes')
    print(f'  Best val F1: {max(history.history["val_f1"]):.4f}')
    print('='*70)
    
    # Memory cleanup
    del history
    gc.collect()
else:
    print('⚠️  Skipping CNN-1D (input shape not compatible)')


TRAINING CNN-1D MODEL...
Model compiled. Starting training...
Epoch 1/50
1317/1321 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8739 - f1: 0.8840 - loss: 0.5143 - precision: 0.8904 - recall: 0.8779
Epoch 1: val_f1 improved from -inf to 0.83209, saving model to /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/neural/cnn1d_best.keras
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8740 - f1: 0.8840 - loss: 0.5139 - precision: 0.8905 - recall: 0.8780 - val_accuracy: 0.7939 - val_f1: 0.8321 - val_loss: 0.6291 - val_precision: 0.7491 - val_recall: 0.9357 - learning_rate: 0.0010
Epoch 2/50
1313/1321 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9171 - f1: 0.9236 - loss: 0.2930 - precision: 0.9294 - recall: 0.9178
Epoch 2: val_f1 did not improve from 0.83209
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.9171 - f1: 0.9236 - loss: 0.2929 - precision: 0.9294 - recall: 0.9178 - val_accuracy: 0.6406 - val_f1: 0.5127 - val_loss: 1.9886 - val_precision: 0.9861 -

### 6.3 Train LSTM

In [21]:
if X_train_cnn is not None:
    print('\nTRAINING LSTM MODEL...')
    print('='*70)
    
    # Build model
    lstm_model = build_lstm(cnn_input_shape)
    
    # Compile
    lstm_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            F1Score(name='f1')
        ]
    )
    
    print('Model compiled. Starting training...')
    start_time = time.time()
    
    # Train
    history = lstm_model.fit(
        X_train_cnn, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_val_cnn, y_val),
        class_weight=class_weight,
        callbacks=get_callbacks('lstm'),
        verbose=1
    )
    
    train_time = time.time() - start_time
    
    # Store
    trained_models['LSTM'] = lstm_model
    training_histories['LSTM'] = history.history
    training_times['LSTM'] = train_time
    
    print(f'\n✓ LSTM training complete in {train_time/60:.1f} minutes')
    print(f'  Best val F1: {max(history.history["val_f1"]):.4f}')
    print('='*70)
    
    # Memory cleanup
    del history
    gc.collect()
else:
    print('⚠️  Skipping LSTM (input shape not compatible)')


TRAINING LSTM MODEL...
Model compiled. Starting training...
Epoch 1/50
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8268 - f1: 0.8369 - loss: 0.7165 - precision: 0.8596 - recall: 0.8155
Epoch 1: val_f1 improved from -inf to 0.80049, saving model to /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/neural/lstm_best.keras
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 46s 34ms/step - accuracy: 0.8269 - f1: 0.8370 - loss: 0.7164 - precision: 0.8597 - recall: 0.8155 - val_accuracy: 0.8005 - val_f1: 0.8005 - val_loss: 0.5940 - val_precision: 0.8814 - val_recall: 0.7332 - learning_rate: 0.0010
Epoch 2/50
1320/1321 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8894 - f1: 0.8971 - loss: 0.4213 - precision: 0.9114 - recall: 0.8833
Epoch 2: val_f1 did not improve from 0.80049
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 46s 35ms/step - accuracy: 0.8894 - f1: 0.8971 - loss: 0.4213 - precision: 0.9113 - recall: 0.8833 - val_accuracy: 0.6545 - val_f1: 0.7555 - val_loss: 0.7557 - val_precision: 0.6155

### 6.4 Train CNN-LSTM Hybrid

In [22]:
if X_train_cnn is not None:
    print('\nTRAINING CNN-LSTM HYBRID MODEL...')
    print('='*70)
    
    # Build model
    hybrid_model = build_cnn_lstm(cnn_input_shape)
    
    # Compile
    hybrid_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            F1Score(name='f1')
        ]
    )
    
    print('Model compiled. Starting training...')
    start_time = time.time()
    
    # Train
    history = hybrid_model.fit(
        X_train_cnn, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_val_cnn, y_val),
        class_weight=class_weight,
        callbacks=get_callbacks('cnn_lstm'),
        verbose=1
    )
    
    train_time = time.time() - start_time
    
    # Store
    trained_models['CNN_LSTM'] = hybrid_model
    training_histories['CNN_LSTM'] = history.history
    training_times['CNN_LSTM'] = train_time
    
    print(f'\n✓ CNN-LSTM training complete in {train_time/60:.1f} minutes')
    print(f'  Best val F1: {max(history.history["val_f1"]):.4f}')
    print('='*70)
    
    # Memory cleanup
    del history
    gc.collect()
else:
    print('⚠️  Skipping CNN-LSTM (input shape not compatible)')


TRAINING CNN-LSTM HYBRID MODEL...
Model compiled. Starting training...
Epoch 1/50
1316/1321 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8427 - f1: 0.8545 - loss: 0.6743 - precision: 0.8644 - recall: 0.8449
Epoch 1: val_f1 improved from -inf to 0.87223, saving model to /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/neural/cnn_lstm_best.keras
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - accuracy: 0.8429 - f1: 0.8546 - loss: 0.6738 - precision: 0.8645 - recall: 0.8450 - val_accuracy: 0.8622 - val_f1: 0.8722 - val_loss: 0.5193 - val_precision: 0.8831 - val_recall: 0.8616 - learning_rate: 0.0010
Epoch 2/50
1320/1321 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9081 - f1: 0.9153 - loss: 0.3909 - precision: 0.9217 - recall: 0.9090
Epoch 2: val_f1 did not improve from 0.87223
1321/1321 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - accuracy: 0.9081 - f1: 0.9153 - loss: 0.3908 - precision: 0.9217 - recall: 0.9090 - val_accuracy: 0.8523 - val_f1: 0.8666 - val_loss: 0.4576 - val_pr

## Section 7: Evaluation on Test Set

In [23]:
print('\nEVALUATING ALL MODELS ON TEST SET...')
print('='*70)

test_results = []

for model_name, model in trained_models.items():
    print(f'\nEvaluating {model_name}...')
    
    # Use correct test data
    if model_name == 'MLP':
        X_test_eval = X_test_mlp
    else:
        X_test_eval = X_test_cnn
    
    # Predictions
    start_time = time.time()
    y_pred_proba = model.predict(X_test_eval, verbose=0)
    inference_time = (time.time() - start_time) / len(X_test_eval) * 1000  # ms per sample
    
    y_pred = (y_pred_proba > 0.5).astype(int).flatten()
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Model size
    model_path = MODELS_DIR / f'{model_name.lower()}_best.keras'
    if model_path.exists():
        model_size_mb = model_path.stat().st_size / (1024 * 1024)
    else:
        model_size_mb = 0
    
    # Store results
    test_results.append({
        'Model': model_name,
        'Accuracy': acc,
        'F1': f1,
        'Precision': precision,
        'Recall': recall,
        'AUC': auc,
        'FNR (%)': (1 - recall) * 100,
        'Inference (ms)': inference_time,
        'Size (MB)': model_size_mb,
        'Params': model.count_params(),
        'Train Time (min)': training_times[model_name] / 60
    })
    
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  F1 Score:  {f1:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  AUC:       {auc:.4f}')
    print(f'  FNR:       {(1-recall)*100:.2f}%')
    print(f'  Inference: {inference_time:.2f} ms/sample')
    print(f'  Size:      {model_size_mb:.2f} MB')

# Create results DataFrame
df_results = pd.DataFrame(test_results)
df_results = df_results.sort_values('F1', ascending=False)

print('\n' + '='*70)
print(f'NEURAL MODELS TEST RESULTS ({feature_type} features):')
print('='*70)
print(df_results.to_string(index=False))
print('='*70)

# Save results
results_csv = RESULTS_DIR / f'07_neural_models_results_{feature_type}.csv'
df_results.to_csv(results_csv, index=False)
print(f'\n✓ Saved results: {results_csv.name}')


EVALUATING ALL MODELS ON TEST SET...

Evaluating MLP...
  Accuracy:  0.8510
  F1 Score:  0.8584
  Precision: 0.8916
  Recall:    0.8276
  AUC:       0.9241
  FNR:       17.24%
  Inference: 0.07 ms/sample
  Size:      36.04 MB

Evaluating CNN1D...
  Accuracy:  0.9725
  F1 Score:  0.9750
  Precision: 0.9690
  Recall:    0.9810
  AUC:       0.9960
  FNR:       1.90%
  Inference: 0.08 ms/sample
  Size:      1.13 MB

Evaluating LSTM...
  Accuracy:  0.9686
  F1 Score:  0.9714
  Precision: 0.9673
  Recall:    0.9755
  AUC:       0.9949
  FNR:       2.45%
  Inference: 0.40 ms/sample
  Size:      2.11 MB

Evaluating CNN_LSTM...
  Accuracy:  0.9779
  F1 Score:  0.9799
  Precision: 0.9753
  Recall:    0.9844
  AUC:       0.9975
  FNR:       1.56%
  Inference: 0.09 ms/sample
  Size:      1.39 MB

NEURAL MODELS TEST RESULTS (3ch features):
   Model  Accuracy       F1  Precision   Recall      AUC   FNR (%)  Inference (ms)  Size (MB)  Params  Train Time (min)
CNN_LSTM  0.977908 0.979855   0.975336 0

## Section 8: Compare with Classical Models

In [24]:
print('\nCOMPARING WITH CLASSICAL MODELS...')
print('='*70)

# Load classical model results from notebook 06a
classical_results_file = RESULTS_DIR / '06a_recommendation_report.json'

if classical_results_file.exists():
    with open(classical_results_file, 'r') as f:
        classical_rec = json.load(f)
    
    print('\nBest Classical Model (from notebook 06a):')
    print(f'  Model:     {classical_rec["primary_model"]["model_name"]}')
    print(f'  Features:  {classical_rec["primary_model"]["feature_set"]}')
    print(f'  Test F1:   {classical_rec["primary_model"]["test_f1"]:.4f}')
    print(f'  Recall:    {classical_rec["primary_model"]["recall"]:.4f}')
    print(f'  Inference: {classical_rec["primary_model"]["inference_ms"]:.2f} ms')
    print(f'  Size:      {classical_rec["primary_model"]["size_mb"]:.2f} MB')
    
    # Get best neural model
    best_neural = df_results.iloc[0]
    
    print(f'\nBest Neural Model (current notebook, {feature_type}):')
    print(f'  Model:     {best_neural["Model"]}')
    print(f'  Test F1:   {best_neural["F1"]:.4f}')
    print(f'  Recall:    {best_neural["Recall"]:.4f}')
    print(f'  Inference: {best_neural["Inference (ms)"]:.2f} ms')
    print(f'  Size:      {best_neural["Size (MB)"]:.2f} MB')
    
    print('\n' + '='*70)
    print('COMPARISON:')
    print('='*70)
    
    f1_diff = best_neural['F1'] - classical_rec['primary_model']['test_f1']
    recall_diff = best_neural['Recall'] - classical_rec['primary_model']['recall']
    inference_diff = best_neural['Inference (ms)'] - classical_rec['primary_model']['inference_ms']
    size_diff = best_neural['Size (MB)'] - classical_rec['primary_model']['size_mb']
    
    print(f'F1 difference:        {f1_diff:+.4f} ({"neural wins" if f1_diff > 0 else "classical wins"})')
    print(f'Recall difference:    {recall_diff:+.4f} ({"neural wins" if recall_diff > 0 else "classical wins"})')
    print(f'Inference difference: {inference_diff:+.2f} ms ({"neural faster" if inference_diff < 0 else "classical faster"})')
    print(f'Size difference:      {size_diff:+.2f} MB ({"neural smaller" if size_diff < 0 else "classical smaller"})')
    
else:
    print('⚠️  Classical model results not found (run notebook 06a first)')
    print('\nBest Neural Model:')
    best_neural = df_results.iloc[0]
    print(f'  Model:     {best_neural["Model"]}')
    print(f'  Test F1:   {best_neural["F1"]:.4f}')
    print(f'  Recall:    {best_neural["Recall"]:.4f}')
    print(f'  Inference: {best_neural["Inference (ms)"]:.2f} ms')
    print(f'  Size:      {best_neural["Size (MB)"]:.2f} MB')

print('='*70)


COMPARING WITH CLASSICAL MODELS...

Best Classical Model (from notebook 06a):
  Model:     XGBoost
  Features:  COMBINED
  Test F1:   0.9877
  Recall:    0.9955
  Inference: 0.01 ms
  Size:      1.57 MB

Best Neural Model (current notebook, 3ch):
  Model:     CNN_LSTM
  Test F1:   0.9799
  Recall:    0.9844
  Inference: 0.09 ms
  Size:      1.39 MB

COMPARISON:
F1 difference:        -0.0078 (classical wins)
Recall difference:    -0.0111 (classical wins)
Inference difference: +0.08 ms (classical faster)
Size difference:      -0.19 MB (neural smaller)


## Section 9: Visualization

In [25]:
print('\nGenerating visualizations...')

# Figure 1: Training history
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for model_name, history in training_histories.items():
    # Loss
    axes[0, 0].plot(history['loss'], label=f'{model_name} Train', alpha=0.7)
    axes[0, 0].plot(history['val_loss'], label=f'{model_name} Val', linestyle='--', alpha=0.7)

    # F1
    axes[0, 1].plot(history['f1'], label=f'{model_name} Train', alpha=0.7)
    axes[0, 1].plot(history['val_f1'], label=f'{model_name} Val', linestyle='--', alpha=0.7)

    # Precision
    axes[1, 0].plot(history['precision'], label=f'{model_name} Train', alpha=0.7)
    axes[1, 0].plot(history['val_precision'], label=f'{model_name} Val', linestyle='--', alpha=0.7)

    # Recall
    axes[1, 1].plot(history['recall'], label=f'{model_name} Train', alpha=0.7)
    axes[1, 1].plot(history['val_recall'], label=f'{model_name} Val', linestyle='--', alpha=0.7)

axes[0, 0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

axes[0, 1].set_title('F1 Score', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('F1')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

axes[1, 0].set_title('Precision', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

axes[1, 1].set_title('Recall', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
fig1_path = FIGURES_DIR / f'07_training_history_{feature_type}.png'
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {fig1_path.name}')
plt.close()


Generating visualizations...
✓ Saved: 07_training_history_3ch.png


In [26]:
# Figure 2: Model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# F1 vs Inference Time
axes[0].scatter(df_results['Inference (ms)'], df_results['F1'], s=200, alpha=0.6)
for i, row in df_results.iterrows():
    axes[0].annotate(row['Model'], (row['Inference (ms)'], row['F1']),
                    textcoords='offset points', xytext=(0, 10), ha='center')
axes[0].set_xlabel('Inference Time (ms/sample)', fontsize=12)
axes[0].set_ylabel('F1 Score', fontsize=12)
axes[0].set_title('Performance vs Speed', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# F1 vs Model Size
axes[1].scatter(df_results['Size (MB)'], df_results['F1'], s=200, alpha=0.6, c='green')
for i, row in df_results.iterrows():
    axes[1].annotate(row['Model'], (row['Size (MB)'], row['F1']),
                    textcoords='offset points', xytext=(0, 10), ha='center')
axes[1].axvline(x=0.5, color='red', linestyle='--', label='500KB Target', alpha=0.7)
axes[1].set_xlabel('Model Size (MB)', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('Performance vs Size', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Recall comparison
axes[2].barh(df_results['Model'], df_results['Recall'], color='steelblue')
axes[2].set_xlabel('Recall (minimize false negatives)', fontsize=12)
axes[2].set_title('Recall Comparison', fontsize=14, fontweight='bold')
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
fig2_path = FIGURES_DIR / f'07_model_comparison_{feature_type}.png'
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {fig2_path.name}')
plt.close()

print('\n✓ All visualizations complete')

✓ Saved: 07_model_comparison_3ch.png

✓ All visualizations complete


## Section 10: Deployment Readiness Assessment

In [27]:
print('\nDEPLOYMENT READINESS ASSESSMENT (ESP32-S3):')
print('='*70)

TARGET_SIZE_MB = 0.5  # 500KB
TARGET_INFERENCE_MS = 50
TARGET_F1 = 0.98
TARGET_RECALL = 0.98

print('\nDeployment Targets:')
print(f'  Model size:     <{TARGET_SIZE_MB} MB (500KB)')
print(f'  Inference time: <{TARGET_INFERENCE_MS} ms')
print(f'  F1 score:       >{TARGET_F1:.2f}')
print(f'  Recall:         >{TARGET_RECALL:.2f} (minimize false negatives)')

print('\n' + '-'*70)
print('Model Readiness:')
print('-'*70)

for _, row in df_results.iterrows():
    model = row['Model']
    
    # Check criteria
    size_ok = row['Size (MB)'] < TARGET_SIZE_MB
    inference_ok = row['Inference (ms)'] < TARGET_INFERENCE_MS
    f1_ok = row['F1'] >= TARGET_F1
    recall_ok = row['Recall'] >= TARGET_RECALL
    
    print(f'\n{model}:')
    print(f'  Size:      {row["Size (MB)"]:.2f} MB {"✓" if size_ok else "✗ (needs quantization)"}')
    print(f'  Inference: {row["Inference (ms)"]:.2f} ms {"✓" if inference_ok else "✗ (too slow)"}')
    print(f'  F1:        {row["F1"]:.4f} {"✓" if f1_ok else "✗ (below target)"}')
    print(f'  Recall:    {row["Recall"]:.4f} {"✓" if recall_ok else "✗ (below target)"}')
    
    all_ok = size_ok and inference_ok and f1_ok and recall_ok
    
    if all_ok:
        print(f'  → READY FOR DEPLOYMENT')
    elif not size_ok and inference_ok and f1_ok and recall_ok:
        print(f'  → QUANTIZATION REQUIRED (performance good)')
    else:
        print(f'  → NEEDS OPTIMIZATION')

print('\n' + '='*70)
print('NEXT STEPS:')
print('='*70)
print('\n1. Model Quantization (Notebook 08):')
print('   - Convert best model to TensorFlow Lite')
print('   - Apply INT8 quantization')
print('   - Target: <500KB model size')
print('   - Validate accuracy after quantization')
print('\n2. Compare 1ch vs 3ch:')
print(f'   - Currently using: {feature_type}')
print(f'   - Re-run with USE_3CH = {not USE_3CH} to compare')
print('   - Evaluate accuracy/size trade-off')
print('\n3. ESP32-S3 Deployment (Notebook 09):')
print('   - Port feature extraction to C/C++')
print('   - Integrate TFLite Micro runtime')
print('   - Real-time testing on device')
print('='*70)


DEPLOYMENT READINESS ASSESSMENT (ESP32-S3):

Deployment Targets:
  Model size:     <0.5 MB (500KB)
  Inference time: <50 ms
  F1 score:       >0.98
  Recall:         >0.98 (minimize false negatives)

----------------------------------------------------------------------
Model Readiness:
----------------------------------------------------------------------

CNN_LSTM:
  Size:      1.39 MB ✗ (needs quantization)
  Inference: 0.09 ms ✓
  F1:        0.9799 ✗ (below target)
  Recall:    0.9844 ✓
  → NEEDS OPTIMIZATION

CNN1D:
  Size:      1.13 MB ✗ (needs quantization)
  Inference: 0.08 ms ✓
  F1:        0.9750 ✗ (below target)
  Recall:    0.9810 ✓
  → NEEDS OPTIMIZATION

LSTM:
  Size:      2.11 MB ✗ (needs quantization)
  Inference: 0.40 ms ✓
  F1:        0.9714 ✗ (below target)
  Recall:    0.9755 ✗ (below target)
  → NEEDS OPTIMIZATION

MLP:
  Size:      36.04 MB ✗ (needs quantization)
  Inference: 0.07 ms ✓
  F1:        0.8584 ✗ (below target)
  Recall:    0.8276 ✗ (below target)
  → 

## Section 11: Save Final Report & Cleanup

In [28]:
print('\nSAVING FINAL REPORT...')
print('='*70)

# Get best neural model
best_neural = df_results.iloc[0]

# Create report
report = {
    'timestamp': datetime.now().isoformat(),
    'notebook': '07_neural_training_evaluation',
    'feature_type': feature_type,
    'best_neural_model': {
        'model_name': best_neural['Model'],
        'test_f1': float(best_neural['F1']),
        'test_accuracy': float(best_neural['Accuracy']),
        'precision': float(best_neural['Precision']),
        'recall': float(best_neural['Recall']),
        'auc': float(best_neural['AUC']),
        'fnr_percent': float(best_neural['FNR (%)']),
        'inference_ms': float(best_neural['Inference (ms)']),
        'size_mb': float(best_neural['Size (MB)']),
        'parameters': int(best_neural['Params']),
        'training_time_minutes': float(best_neural['Train Time (min)'])
    },
    'total_models_trained': len(trained_models),
    'deployment_status': 'Quantization required for ESP32-S3',
    'next_steps': [
        'Quantize best model to TensorFlow Lite INT8',
        'Target <500KB model size',
        f'Compare with {"1ch" if feature_type == "3ch" else "3ch"} features',
        'Port to ESP32-S3 with TFLite Micro',
        'Real-world field testing'
    ]
}

# Save JSON report
report_json = RESULTS_DIR / f'07_neural_recommendation_report_{feature_type}.json'
with open(report_json, 'w') as f:
    json.dump(report, f, indent=2)

print(f'✓ Saved JSON report: {report_json.name}')

# Save best model explicitly
best_model = trained_models[best_neural['Model']]
best_model_path = MODELS_DIR / f'best_neural_model_{feature_type}.keras'
best_model.save(best_model_path)
print(f'✓ Saved best model: {best_model_path.name}')

print('\n' + '='*70)
print('✓ NOTEBOOK 07 COMPLETE')
print('='*70)
print(f'\nBest Neural Model ({feature_type}): {best_neural["Model"]}')
print(f'  Test F1:   {best_neural["F1"]:.4f}')
print(f'  Recall:    {best_neural["Recall"]:.4f}')
print(f'  Inference: {best_neural["Inference (ms)"]:.2f} ms')
print(f'  Size:      {best_neural["Size (MB)"]:.2f} MB')
print('\nReady for quantization in Notebook 08!')
print('='*70)

# Final memory cleanup
del X_train_norm, X_val_norm, X_test_norm
if X_train_cnn is not None:
    del X_train_cnn, X_val_cnn, X_test_cnn
gc.collect()
K.clear_session()
print('\n✓ Memory cleaned up')


SAVING FINAL REPORT...
✓ Saved JSON report: 07_neural_recommendation_report_3ch.json
✓ Saved best model: best_neural_model_3ch.keras

✓ NOTEBOOK 07 COMPLETE

Best Neural Model (3ch): CNN_LSTM
  Test F1:   0.9799
  Recall:    0.9844
  Inference: 0.09 ms
  Size:      1.39 MB

Ready for quantization in Notebook 08!

✓ Memory cleaned up
